# Arxiv API Data Collection Script

In [10]:
import json
import time
import re
from datetime import datetime
from typing import Dict, List, Optional, Set
from urllib.parse import urlencode
from urllib.request import urlopen, Request
import xml.etree.ElementTree as ET

ARXIV_API_URL = "https://export.arxiv.org/api/query"

# Namespaces used by the arXiv Atom feed
NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

CATEGORIES = ["cs.LG", "cs.CL", "cs.AI", "stat.ML"]
START_YEAR = 2015
END_YEAR = 2026
TARGET_TOTAL = 15000
BATCH_SIZE = 200  # conservative size
SLEEP_SECONDS = 3.0  # arXiv examples recommend ~3 seconds between calls

OUTPUT_JSONL = "data/papers.jsonl"
OUTPUT_META = "data/dataset_metadata.json"


def build_submitted_date_range(start_year: int, end_year: int) -> str:
    # arXiv expects submittedDate as YYYYMMDDHHMM
    start = f"{start_year}01010000"
    end = f"{end_year}12312359"
    return f"[{start} TO {end}]"


def build_search_query(categories: List[str], start_year: int, end_year: int) -> str:
    # Example: (cat:cs.LG OR cat:cs.CL OR cat:cs.AI OR cat:stat.ML) AND submittedDate:[201501010000 TO 202612312359]
    category_query = " OR ".join(f"cat:{c}" for c in categories)
    submitted_date_range = build_submitted_date_range(start_year, end_year)
    return f"({category_query}) AND submittedDate:{submitted_date_range}"


def normalise_whitespace(text: Optional[str]) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def extract_arxiv_id(entry_id: str) -> str:
    # Example entry_id: http://arxiv.org/abs/2104.08663v1
    arxiv_id = entry_id.rsplit("/", 1)[-1]
    # keep versionless id for deduplication
    return arxiv_id.split("v")[0]


def parse_entry(entry: ET.Element) -> Optional[Dict]:
    entry_id = entry.findtext("atom:id", default="", namespaces=NS)
    title = normalise_whitespace(entry.findtext("atom:title", default="", namespaces=NS))
    summary = normalise_whitespace(entry.findtext("atom:summary", default="", namespaces=NS))
    published = entry.findtext("atom:published", default="", namespaces=NS)
    updated = entry.findtext("atom:updated", default="", namespaces=NS)

    if not entry_id or not title or not summary or not published:
        return None

    arxiv_id = extract_arxiv_id(entry_id)

    # Authors
    authors = []
    for author in entry.findall("atom:author", NS):
        name = author.findtext("atom:name", default="", namespaces=NS).strip()
        if name:
            authors.append(name)

    # Primary category
    primary_category_el = entry.find("arxiv:primary_category", NS)
    primary_category = primary_category_el.attrib.get("term", "") if primary_category_el is not None else ""

    # All categories
    categories = []
    for cat in entry.findall("atom:category", NS):
        term = cat.attrib.get("term", "")
        if term:
            categories.append(term)

    # PDF URL if present
    pdf_url = None
    for link in entry.findall("atom:link", NS):
        if link.attrib.get("title") == "pdf":
            pdf_url = link.attrib.get("href")
            break

    try:
        published_year = datetime.fromisoformat(published.replace("Z", "+00:00")).year
    except ValueError:
        return None

    return {
        "paper_id": arxiv_id,
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": summary,
        "authors": authors,
        "primary_category": primary_category,
        "all_categories": categories,
        "published_date": published,
        "updated_date": updated,
        "published_year": published_year,
        "source": "arXiv",
        "entry_url": entry_id,
        "pdf_url": pdf_url,
    }


def fetch_batch(search_query: str, start: int, max_results: int) -> List[Dict]:
    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "ascending",
    }
    url = f"{ARXIV_API_URL}?{urlencode(params)}"
    print(f"Request URL: {url}")

    req = Request(
        url,
        headers={
            "User-Agent": "AyomideMScProject/1.0 (academic data collection)"
        },
    )

    with urlopen(req, timeout=60) as response:
        xml_data = response.read()

    root = ET.fromstring(xml_data)
    entries = root.findall("atom:entry", NS)

    records = []
    for entry in entries:
        record = parse_entry(entry)
        if record is not None:
            records.append(record)

    return records


def collect_arxiv_papers(
    categories: List[str],
    start_year: int,
    end_year: int,
    target_total: int,
    batch_size: int,
    sleep_seconds: float,
) -> List[Dict]:
    query = build_search_query(categories, start_year, end_year)

    papers: List[Dict] = []
    seen_ids: Set[str] = set()
    start = 0

    while len(papers) < target_total:
        batch = fetch_batch(query, start=start, max_results=batch_size)

        if not batch:
            break

        added_this_batch = 0
        for paper in batch:
            if not (start_year <= paper["published_year"] <= end_year):
                continue
            if paper["paper_id"] in seen_ids:
                continue

            seen_ids.add(paper["paper_id"])
            papers.append(paper)
            added_this_batch += 1

            if len(papers) >= target_total:
                break

        print(
            f"Fetched batch starting at {start}: "
            f"{len(batch)} records, added {added_this_batch}, total kept {len(papers)}"
        )

        start += batch_size
        time.sleep(sleep_seconds)

    return papers


def write_jsonl(records: List[Dict], filepath: str) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_metadata(records: List[Dict], categories: List[str], start_year: int, end_year: int) -> Dict:
    by_category = {}
    for c in categories:
        by_category[c] = sum(1 for r in records if r["primary_category"] == c)

    avg_abstract_words = round(
        sum(len(r["abstract"].split()) for r in records) / len(records), 2
    ) if records else 0

    return {
        "source": "arXiv API",
        "collection_timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "categories": categories,
        "year_range": {"start": start_year, "end": end_year},
        "corpus_size": len(records),
        "average_abstract_length_words": avg_abstract_words,
        "category_counts": by_category,
        "fields": [
            "paper_id",
            "arxiv_id",
            "title",
            "abstract",
            "authors",
            "primary_category",
            "all_categories",
            "published_date",
            "updated_date",
            "published_year",
            "source",
            "entry_url",
            "pdf_url",
        ],
    }


def main() -> None:
    papers = collect_arxiv_papers(
        categories=CATEGORIES,
        start_year=START_YEAR,
        end_year=END_YEAR,
        target_total=TARGET_TOTAL,
        batch_size=BATCH_SIZE,
        sleep_seconds=SLEEP_SECONDS,
    )

    write_jsonl(papers, OUTPUT_JSONL)

    metadata = build_metadata(papers, CATEGORIES, START_YEAR, END_YEAR)
    with open(OUTPUT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(papers)} papers to {OUTPUT_JSONL}")
    print(f"Saved metadata to {OUTPUT_META}")


if __name__ == "__main__":
    main()

Request URL: https://export.arxiv.org/api/query?search_query=%28cat%3Acs.LG+OR+cat%3Acs.CL+OR+cat%3Acs.AI+OR+cat%3Astat.ML%29+AND+submittedDate%3A%5B201501010000+TO+202612312359%5D&start=0&max_results=200&sortBy=submittedDate&sortOrder=ascending
Fetched batch starting at 0: 200 records, added 200, total kept 200
Request URL: https://export.arxiv.org/api/query?search_query=%28cat%3Acs.LG+OR+cat%3Acs.CL+OR+cat%3Acs.AI+OR+cat%3Astat.ML%29+AND+submittedDate%3A%5B201501010000+TO+202612312359%5D&start=200&max_results=200&sortBy=submittedDate&sortOrder=ascending
Fetched batch starting at 200: 200 records, added 200, total kept 400
Request URL: https://export.arxiv.org/api/query?search_query=%28cat%3Acs.LG+OR+cat%3Acs.CL+OR+cat%3Acs.AI+OR+cat%3Astat.ML%29+AND+submittedDate%3A%5B201501010000+TO+202612312359%5D&start=400&max_results=200&sortBy=submittedDate&sortOrder=ascending
Fetched batch starting at 400: 200 records, added 200, total kept 600
Request URL: https://export.arxiv.org/api/query?se

HTTPError: HTTP Error 500: Internal Server Error